# 🎮 Reinforcement Learning for Connect-4
## 🧠 An N-Tuple TD(λ) Agent from Scratch

In this workshop we train a **Temporal-Difference (TD) agent** to play Connect-4 to a strong level — without any hand-crafted evaluation function, without minimax search during training, and without labelled game data.

The agent learns purely from **self-play**: it plays games, receives a reward signal (+1 win / 0 draw / −1 loss), and gradually improves its value function.

---

### 🔴 What is Connect-4?

<div align="center">
  <img src="img/c4_board.png" alt="Connect-4 board" width="400"/>

  <p><em>Figure 1: The Connect‑4 board.</em></p>
</div>

Connect-4 is a two-player, perfect-information, zero-sum board game:

- **Board**: 7 columns × 6 rows, tokens fall to the lowest empty cell in a column
- **Goal**: be the first to place **4 tokens in a row** — horizontally, vertically, or diagonally
- **Players alternate** turns; Yellow (player 1) moves first

---

### 🤔 Why Connect-4 for RL?

| Property | Why it matters |
|---|---|
| Fully deterministic | No hidden information — a clean testbed for value learning |
| Compact state | Entire board fits in 2 × 64-bit integers (bitboard) |
| Non-trivial | Far too large for brute-force; strategy is genuinely required |
| Solved | Ground-truth exists — we can measure how good our agent is |

---

### 📐 The Scale of the Problem

| Fact | Value |
|---|---|
| Total positions | **4,531,985,219,092** (~4.5 trillion) |
| Weak solution | **James Allen** and **Victor Allis** solved it independently in **1988** — just ~two weeks apart! Yellow always wins with perfect play, but a full proof was out of reach at the time |
| Strong solution | **7 years later**, **John Tromp** (Netherlands) completed a full database proof on Sun Microsystems & Silicon Graphics workstations — **40,000 combined CPU-hours** |
| Brute-force at 1,000 pos/s | Would take **~143 years** |
| Brute-force at 1,000,000 pos/s | Still takes **~52 days** |
| Positions per human on Earth | **>500** — more C4 positions than 500× the world's population |

> Our agent cannot search the whole tree. Instead it learns a compact **value function** that scores any position in microseconds.

---

### 🗺️ Today's Journey

You already saw the general reinforcement-learning framework in **Lab 1**. This lab grounds those ideas in a concrete, GPU-friendly implementation and walks through every building block of a TD(λ) agent for Connect-4:

| Section | What you'll see | Key widget |
|---|---|---|
| **§1** Environment | How the board is encoded in 64-bit integers and how `BoardBatch` runs thousands of games in parallel | Bitboard overlay explorer |
| **§2** State | Minimal state representation, sparse rewards, afterstates | Reward-timeline; afterstate equivalence |
| **§3** Agent & Policy | N-Tuple Network evaluator + ε-greedy afterstate selection | N-Tuple visualizer; ε-greedy explorer |
| **§4** Update Rule | TD error → TD(0) update → n-step returns → TD(λ) → full-game walkthrough | TD training-step visualizer; λ-return explorer |
| **§5** Does it work? | Compare untrained vs. trained checkpoints, then play a trained agent against BitBully | Checkpoint comparison + rematch widget |

---

### 🎯 Before we start: face the boss

<div align="center">
  <img src="img/rl-loop.png" alt="RL interaction loop" width="800"/>

  <p><em>Figure 2: The reinforcement learning interaction loop between agent and environment.</em></p>
</div>

Play a couple of games against **BitBully** (a 4-ply minimax agent) in the widget below. If you can't beat it — don't worry. By the end of this notebook you'll have watched a *self-taught* N-Tuple TD(λ) agent go head-to-head with it.

## 🕹️ Play Connect-4 vs. BitBully

You play as **Yellow** — click a column to drop your token. BitBully plays Red with a 4-ply minimax search.

> **Tip:** Try to win — it's harder than it looks! Remember the feeling of losing. Your reward-driven N-Tuple agent at the end of §4 has to beat this same opponent without ever being told a single rule of good play.

In [ ]:
%matplotlib ipympl
from bitbully import BitBully
from bitbully.gui_c4 import GuiC4

# Human (Yellow) vs BitBully 4-ply (Red)
agents = {
    "BitBully (4-ply)": BitBully(opening_book=None, tie_break="center", max_depth=4),
}

c4gui = GuiC4(agents=agents, autoplay=True)
display(c4gui.get_widget())

## 🌍 1. Environment

> 🔧 **Implementation detail — safe to skim.** §1 is an *optional deep-dive* into how the board is represented and how move generation / win detection are implemented using bit tricks. If you're mostly interested in the RL story (state, rewards, policy, TD update), you can jump straight to **§2 State**. The bit tricks are beautiful, but not required to follow §2–§4.

The **environment** is the Connect-4 game itself. It is fully deterministic: given a state and an action, the next state is uniquely determined — no stochasticity (unlike, e.g., Backgammon or card games).

The environment is implemented as `BoardBatch` — a vectorized, PyTorch-native simulator that runs **thousands of games in parallel** on CPU or GPU. What information it stores and what that means in the RL/MDP sense is the subject of **§2 State**. This section focuses on the *mechanics*: how the board is encoded and how tokens are placed, players are swapped, and wins are detected — all using bitwise operations on integers.

### 🔲 1.1 Bitboards

Each board position is encoded as a **single 64-bit integer**. The 7 columns each occupy 9 bits (6 playable rows + 3 sentinel bits):

```
Bit indices in a uint64 (9 bits per column, 7 columns):

         C0    C1    C2    C3    C4    C5    C6
       ┌─────┬─────┬─────┬─────┬─────┬─────┬─────┐
  S2   │  8  │ 17  │ 26  │ 35  │ 44  │ 53  │ 62  │  ← sentinel
  S1   │  7  │ 16  │ 25  │ 34  │ 43  │ 52  │ 61  │  ← sentinel
  S0   │  6  │ 15  │ 24  │ 33  │ 42  │ 51  │ 60  │  ← sentinel
       ├─────┼─────┼─────┼─────┼─────┼─────┼─────┤
  R5   │  5  │ 14  │ 23  │ 32  │ 41  │ 50  │ 59  │  ← top row
  R4   │  4  │ 13  │ 22  │ 31  │ 40  │ 49  │ 58  │
  R3   │  3  │ 12  │ 21  │ 30  │ 39  │ 48  │ 57  │
  R2   │  2  │ 11  │ 20  │ 29  │ 38  │ 47  │ 56  │
  R1   │  1  │ 10  │ 19  │ 28  │ 37  │ 46  │ 55  │
  R0   │  0  │  9  │ 18  │ 27  │ 36  │ 45  │ 54  │  ← bottom row
       └─────┴─────┴─────┴─────┴─────┴─────┴─────┘
```

**Why sentinels?** Legal move generation uses an addition trick. Adding `BB_BOTTOM_ROW` to `all_tokens` propagates carry bits upward through a column. The 3 sentinel bits absorb carry from a full column, preventing it from overflowing into the next column.

$$\text{legal\_moves} = (\texttt{all\_tokens} + \texttt{BB\_BOTTOM\_ROW}) \mathbin{\&} \texttt{BB\_ALL\_LEGAL}$$

### 👁️ 1.2 Visualization Widget

The `BitboardVisualizer` renders the raw bitboard as an interactive Connect-4 board.
Click any column to drop a token and watch the bitboards update in real time.

Available **overlays** (select from the dropdowns):

| Overlay | What it shows |
|---|---|
| `all_tokens` | Every piece on the board — both players combined |
| `active_tokens` | Only the current player's pieces |
| `opponent_tokens` | Derived via XOR — never stored explicitly |
| `BB_BOTTOM_ROW` | Constant seed for the carry-propagation trick (one bit per column at row 0) |
| `all_tokens + BB_BOTTOM_ROW` | Carry propagation result — landing bits before masking |
| `BB_ALL_LEGAL` | Constant mask covering all 42 playable cells |
| `legal_moves` | Carry-propagation trick in action |
| `winning_positions` | Cells where a token completes four-in-a-row (threats highlighted) |
| `Bit Index Map` | Which bit in the `int64` corresponds to each cell |

> Use the overlays as you read the sections below — they illustrate every operation interactively.

In [ ]:
%matplotlib ipympl
from techdays26.gui_bitboard import BitboardVisualizer

# Launch an interactive board — click columns to play, select overlay from the dropdown
vis = BitboardVisualizer()
vis.show()

### ⚙️ 1.3 Basic Operations

Three fundamental operations drive the game loop:

#### 🪨 Set stones — `play_columns` / `play_masks`

Placing a token uses the carry-propagation trick introduced in §1.1.
`play_columns(cols)` finds the lowest empty bit in the chosen column and sets it in `all_tokens`.

Select the **`legal_moves`** overlay in the widget above to see the carry trick in action.

In [ ]:
import torch
from techdays26.torch_board import BoardBatch

board = BoardBatch.empty(1, device="cpu")
for col in [3, 3, 3]:
    board.play_columns(torch.tensor([col]))

print("After 3 tokens in column 3:")
print(f"  all_tokens col-3 bits: {(int(board.all_tokens[0]) >> 27) & 0b111111:06b}")
print(f"  moves_left = {int(board.moves_left[0])}")

#### 🔄 Switch players — a single XOR

After every move, `play_columns` / `play_masks` executes:

```python
active_tokens ^= all_tokens
```

Because `all_tokens` now includes the token just placed, XOR-ing with the *previous* `active_tokens` yields the **opponent's** pieces — which become the new `active_tokens` for the next turn. No copy, no extra field.

> Switch between the **`active_tokens`** and **`opponent_tokens`** overlays in the widget to see this in action.

#### 🏆 Detect wins — shift-AND in 4 directions

`has_win()` checks whether the player who just moved has four-in-a-row using the **shift-AND** technique: shift a bitboard by the stride for a given direction, AND with the original, repeat.

| Direction  | Bit stride |
|------------|-----------|
| Horizontal | 9 (column offset) |
| Vertical   | 1 |
| Diagonal ↗ | 10 (col + 1) |
| Diagonal ↘ | 8  (col − 1) |

For any direction with bit-stride *s*:
```python
pairs = y & (y >> s)            # bit i set ↔ tokens at i AND i+s
win   = pairs & (pairs >> 2*s)  # bit i set ↔ four consecutive tokens
```

The entire win check runs in **O(1)** — no loop over cells.

> Select the **`winning_positions`** overlay in the widget to see threats highlighted live.

### 🚀 1.4 Batched Simulation — Thousands of Games in Parallel

So far every example used a single board (`BoardBatch.empty(1, ...)`). In training, the agent plays **thousands of games simultaneously** — each step advances *all* boards at once.

`BoardBatch` stores a flat batch of `B` boards as three tensors of shape `[B]`. Every operation — `play_columns`, `has_win`, `reward`, `reset` — is a vectorised PyTorch op that processes all `B` boards in one call, with no Python loop over individual games.

When a game finishes (win or draw), `reset()` clears just that board back to the empty position — the other games keep running. This means the agent always has `B` active games, and finished games are immediately recycled.

In [ ]:
import torch
from techdays26.torch_board import BoardBatch

B = 5  # 5 parallel games (training uses B = 20,000)
board = BoardBatch.empty(B, device="cpu")

# Each board gets a DIFFERENT random legal move
legal = board.generate_legal_moves()
# Pick one random legal move per board using reservoir sampling over move masks
chosen = torch.zeros(B, dtype=torch.int64)
for mv in board.iter_move_masks(legal):
    if not (mv != 0).any():
        break
    # For each board where this move is legal, randomly decide whether to keep it
    mask = mv != 0
    chosen = torch.where(mask & (torch.rand(B) < 0.5), mv, chosen)
# Fallback: if any board has no chosen move yet, pick the first legal move
chosen = torch.where(chosen == 0, legal & (-legal), chosen)
board.play_masks(chosen)

cols = [(int(c).bit_length() - 1) // 9 if c != 0 else -1 for c in chosen.tolist()]
print(f"Batch size: {board.all_tokens.shape[0]} boards")
print(f"Columns played: {cols}")
print(f"moves_left:     {board.moves_left.tolist()}  (all 41 — one move each)")

# Play a few more rounds with random legal moves
for _ in range(5):
    legal = board.generate_legal_moves()
    first_legal = legal & (-legal)  # pick lowest legal move per board
    board.play_masks(first_legal)

# Check which games are done
done = board.done()
wins = board.has_win()
print(f"\nAfter 6 moves per game:")
print(f"  done:    {done.tolist()}")
print(f"  has_win: {wins.tolist()}")

# Reset finished games — others keep playing
board.reset(done)
print(
    f"  moves_left after reset: {board.moves_left.tolist()}  (reset boards \u2192 42)"
)

Try the interactive version — click *Step* to watch 8 games unfold in parallel:

In [ ]:
%matplotlib ipympl
from techdays26.gui_batch import BatchSimulationVisualizer

vis = BatchSimulationVisualizer(n_boards=8)
vis.show()

### ⚡ 1.5 Advantages of the Bitboard Representation

| Property | Benefit |
|---|---|
| **Compact** | Entire game state = 2 × int64 + 1 × int16 — fits in a few registers |
| **Vectorized** | `BoardBatch` stores a whole batch `[B]` of boards as three flat tensors; all operations are elementwise PyTorch ops |
| **GPU-ready** | One `.to("cuda")` call moves the entire batch; every bitwise op runs as a fused GPU kernel with uniform control flow across all boards |
| **O(1) move generation** | One addition + AND computes all 7 landing squares — no loop over columns |
| **O(1) win check** | Four shift-AND operations cover all four directions simultaneously |
| **No copies for player swap** | `active_tokens ^= all_tokens` — one instruction, in-place |
| **Scalable** | Training uses `B = 20 000` parallel games; the bitboard layout makes this practical |

> **Questions — §1 Environment**
>
> 1. Why does each column use **9 bits** instead of just 6? What would go wrong with only 6?
> 2. The XOR player swap (`active_tokens ^= all_tokens`) works because of a specific invariant. What is it?
> 3. The win check uses 4 different bit strides. Why is the horizontal stride 9 and not 1?

<details>
<summary><b>💡 Answers — click to reveal</b></summary>

1. **9 bits per column.** The extra 3 bits are *sentinels* that absorb carries during the legal-move calculation `(all_tokens + BB_BOTTOM_ROW) & BB_ALL_LEGAL`. Without them, a carry from a full column would overflow into the next column's bits and corrupt them. With 9 bits (6 playable + 3 sentinel), the carry is trapped inside the column and masked away by `BB_ALL_LEGAL`.

2. **XOR invariant.** `active_tokens` is always a *subset* of `all_tokens` — the active player's pieces are a subset of all pieces on the board. Therefore `active_tokens XOR all_tokens` leaves exactly the bits that are in `all_tokens` but not in `active_tokens` — i.e., the opponent's pieces. After placing a token into `all_tokens`, that XOR also flips the subset relation: what was the opponent is now the active player.

3. **Horizontal stride = 9.** In the bit layout, row $r$ of column $c$ is at bit index $9c + r$. Moving one cell horizontally means moving one *column* to the right — that is, +9 bits, not +1. Vertical steps are +1 (adjacent rows in the same column); diagonals are +8 and +10 (combining ±1 column with ±1 row).

</details>

## 🧩 2. State

§1 addressed *how* the game is encoded and simulated — the bitboard layout, token placement, player swap, and win detection.

§2 asks a different question: what is the minimal information that constitutes a **state** in the MDP sense — everything the agent needs to act and be evaluated?

For Connect-4 the answer is exactly three integers:

| Field | Tensor shape | Description |
|---|---|---|
| `all_tokens` | `int64 [B]` | Every piece on the board — both players combined |
| `active_tokens` | `int64 [B]` | The *current player's* pieces only |
| `moves_left` | `int16 [B]` | Remaining moves before the board is full (`42 − tokens placed`) |

These three numbers are **complete**: legal moves, win status, whose turn it is, and reward are all uniquely determined by them. There is no hidden information.

The opponent's pieces are not stored — they are derived on demand: $\text{opponent} = \texttt{active\_tokens} \oplus \texttt{all\_tokens}$

---

### 🔚 Terminal States

| Outcome | Condition |
|---|---|
| Win | The side that just moved completed four-in-a-row |
| Loss | The other side just completed four-in-a-row |
| Draw | Board full, no winner |

We will discuss rewards below!

### 2.1 Inspecting a State


In [ ]:
import torch
from techdays26.torch_board import BoardBatch

# Yellow c3, Red c3, Yellow c4, Red c2 — 4 tokens placed
board = BoardBatch.empty(1, device="cpu")
for col in [3, 3, 4, 2]:
    board.play_columns(torch.tensor([col]))

print("State after cols 3, 3, 4, 2:")
print(f"  all_tokens    = {int(board.all_tokens[0])}")
print(f"  active_tokens = {int(board.active_tokens[0])}  (Yellow — moves next)")
print(f"  moves_left    = {int(board.moves_left[0])}")
print(f"  done()        = {bool(board.done()[0])}")
print(f"  reward()      = {float(board.reward()[0]):.0f}")

### 2.2 💰 Rewards

Connect-4 uses **sparse terminal rewards** — the agent receives a signal only at the end of the game.

The reward convention is **global, from Yellow’s perspective** (Yellow-positive):

| Outcome | `reward()` | Meaning |
|---|---|---|
| Yellow wins | `+1` | Good for Yellow |
| Red wins | `−1` | Bad for Yellow |
| Draw (board full, no winner) | `0` | Neutral |
| Game still running | `NaN` | Non-terminal — no reward yet |

Every intermediate step returns `NaN` (not-a-number). This sparsity is what makes the problem hard: the agent must learn to assign credit to early moves that contributed to a win or loss many steps later — the classic **credit assignment problem**.

> **Why no intermediate rewards?** Connect-4 has no natural mid-game progress signal. Any shaping reward (e.g. “reward for three in a row”) would encode human domain knowledge — exactly what we want to avoid.

> **Why Yellow-positive?** Both players’ LUT sets output values on the same scale: $V > 0$ = good for Yellow, $V < 0$ = good for Red. This keeps the value function consistent — the network never needs to flip signs internally.


In [ ]:
# A 42-move drawing line that fills the whole board
draw_sequence = [
    3,
    3,
    3,
    3,
    3,
    1,
    1,
    1,
    1,
    5,
    5,
    5,
    5,
    1,
    5,
    3,
    2,
    2,
    2,
    2,
    2,
    1,
    0,
    6,
    0,
    0,
    0,
    0,
    0,
    6,
    6,
    6,
    4,
    4,
    4,
    4,
    4,
    6,
    6,
    4,
    5,
]

scenarios = [
    ("Yellow wins (vertical)", [3, 6, 3, 6, 3, 6, 3]),  # Yellow col3×4, Red parks col6
    ("Yellow wins (horizontal)", [0, 6, 1, 6, 2, 6, 3]),  # Yellow cols 0-3 bottom row
    ("Red wins (vertical)", [3, 4, 3, 4, 3, 4, 0, 4]),  # Red col4×4, Yellow scattered
    ("Draw (full board)", draw_sequence),
    ("In-progress", [3, 4]),
]

# Yellow-positive reward convention: +1 = Yellow wins, -1 = Red wins, 0 = draw, NaN = running
print(f"  {'Scenario':<28} {'is_done':>8} {'has_win':>8} {'reward':>8}  interpretation")
print("  " + "-" * 76)
for label, seq in scenarios:
    b = BoardBatch.empty(1, device="cpu")
    for col in seq:
        if not bool(b.done()[0]):
            b.play_columns(torch.tensor([col]))
    done = bool(b.done()[0])
    has_win = bool(b.has_win()[0])
    r = float(b.reward()[0])

    # Reward convention is GLOBAL, from Yellow's perspective:
    #   +1 → Yellow wins   -1 → Red wins   0 → Draw   NaN → game running
    if not done:
        note = "game running (NaN)"
    elif r > 0:
        note = "+1 → Yellow wins"
    elif r < 0:
        note = "-1 → Red wins"
    else:
        note = " 0 → Draw"
    r_str = "   nan" if r != r else f"{r:>6.0f}"
    print(f"  {label:<28} {str(done):>8} {str(has_win):>8} {r_str}  {note}")

Step through a random game to see just how **sparse** the reward signal is — and why a learned value function is essential:

In [ ]:
%matplotlib ipympl
from techdays26.gui_reward import RewardTimelineVisualizer

vis = RewardTimelineVisualizer(
    model_path="../exp_20260228_13-46/repeat_0/step_500_model_weights.pt"
)
vis.show()

### 2.3 🔀 State Transition

The transition function $T(s, a) \rightarrow s'$ is **deterministic**: each (state, action) pair maps to exactly one successor state.

`play_columns(col)` performs the transition in-place in four steps:

| Step | Operation | Effect |
|---|---|---|
| 1 | `landing = (all_tokens + col_bottom) & col_mask` | Carry trick finds the landing bit |
| 2 | `all_tokens \|= landing` | New token added to the board |
| 3 | `active_tokens ^= all_tokens` | Perspective swapped to the opponent |
| 4 | `moves_left -= 1` | Move counter decremented |

> Try it in the **widget** (§1.2): click a column and watch `all_tokens` and `active_tokens` update in real time.

### 2.4 🔮 Afterstates

After a move, the board is in an **afterstate** — the position resulting from that move. In self-play, each training step applies exactly **one move** (by whichever player is to move), and the network evaluates the afterstate before and after. The afterstate produced by one training step becomes the "before" position of the next.

> 💡 Our agent learns a value function $V(s')$ over afterstates, rather than a state-action value $Q(s, a)$. Why?

#### Many-to-one compression

Different (state, action) pairs can produce the **same afterstate** — different game histories that converge to the same board position. A $Q$-function would maintain separate entries for each path; the afterstate $V$ shares a single value for all of them, learning faster from less data.

#### Simpler bootstrapping

With afterstates, the TD target is simply:

$$\text{target} = r + V(s'_{\text{next}})$$

No $\max_a Q(s', a)$ inside the bootstrap — the maximisation happens *outside*, during action selection (§3), where we pick the move whose afterstate has the best value.


Explore concrete examples: two different (state, action) pairs that produce the **same afterstate** — illustrating why V(s') is more efficient than Q(s, a):

In [ ]:
%matplotlib ipympl
from techdays26.gui_afterstate_equiv import AfterstateEquivVisualizer

vis = AfterstateEquivVisualizer(
    model_path="../exp_20260228_13-46/repeat_0/step_500_model_weights.pt"
)
vis.show()

> **Questions — §2 State**
>
> 1. The state consists of only 3 integers. How would you recover **whose turn** it is from these?
> 2. Why are intermediate rewards always 0? What could go wrong if you added a reward for "three in a row"?
> 3. Suppose two different game histories produce the same `(all_tokens, active_tokens, moves_left)` triple. Are these the same state or different states? Why does this matter for learning?
> 4. Give an intuitive explanation why a Q(s, a) function would be wasteful for Connect-4 compared to an afterstate value function V(s').

<details>
<summary><b>💡 Answers — click to reveal</b></summary>

1. **Whose turn is it?** Yellow moves first, so after $k$ moves, Yellow is to move iff $k$ is even. Since `moves_left = 42 − k`, we get `yellow_to_move = (moves_left % 2) == 0`. No extra field needed.

2. **No intermediate rewards.** Connect-4 has no natural mid-game progress signal, and any shaping reward like "+0.1 for three in a row" would encode *our* intuition of what's valuable. Three-in-a-row can easily be blocked, so such a reward could mislead the agent into building threats it can't actually convert. The beauty of TD learning is that the agent *discovers* intermediate value on its own, via bootstrapping from the only ground truth we have: the final game outcome.

3. **Same state.** The MDP is Markovian — the state contains everything needed to decide optimal play. Two histories that produce the same triple are, as far as the agent is concerned, indistinguishable. This is precisely why the value function can *generalise*: a position reached via a tactical swindle and the same position reached via a quiet opening get the same $V$, and the network learns from both.

4. **Q vs. V(afterstate).** With Q, you store a value for every `(state, action)` pair. But many different `(s, a)` pairs lead to exactly the same resulting board — different opening-move orders converge to the same position. Q wastes capacity by learning the same answer multiple times, one per path. V(afterstate) collapses all of them into a single entry and gets the benefit of every visit.

</details>

## 🤖 3. Agent & Policy

The agent needs three things to act:

1. A way to enumerate its **legal moves** (§3.1)
2. A **value function** that scores any board position (§3.2–§3.3)
3. A **policy** that uses those scores to pick a move (§3.4)

This section covers all three. We renumber the original sections so that action enumeration is the first sub-section of Agent & Policy — the legal-move generator is not really its own standalone chapter, it's the entry point of the agent's decision loop.

### 3.1 Legal Moves — From State to Available Actions

Before the agent can pick a move, it needs to know which columns are **legal** (not yet full). In §1 we saw the one-line bit trick that returns all legal moves at once:

$$\text{legal\_moves} = (\texttt{all\_tokens} + \texttt{BB\_BOTTOM\_ROW}) \mathbin{\&} \texttt{BB\_ALL\_LEGAL}$$

The result is a bitboard with **one bit set per legal column** — the landing square where the token would fall. `iter_move_masks(moves)` then yields one-hot bitboards for each of those landing bits using the low-bit extraction trick:

```python
mv = moves & -moves   # extract lowest set bit
moves ^= mv           # clear it
```

This is the loop the ε-greedy policy (§4.3) uses to evaluate each afterstate and pick the best move.

In [ ]:
import torch
from techdays26.torch_board import BoardBatch

board = BoardBatch.empty(1, device="cpu")
for col in [3, 3, 3, 3, 3, 3]:  # fill column 3 completely
    board.play_columns(torch.tensor([col]))

legal = board.generate_legal_moves()
legal_int = int(legal[0])

# Print the legal moves bitboard — 9 bits per column, 7 columns
# Format: | C6 | C5 | C4 | C3 | C2 | C1 | C0 |  (each 9 bits wide)
chunks = [f"{(legal_int >> (c * 9)) & 0x1FF:09b}" for c in range(6, -1, -1)]
print(f"legal_moves = {' | '.join(chunks)}")
print(f"               {'   '.join(f'C{c}' for c in range(6, -1, -1))}")

# Extract which columns have a legal move
open_cols = [c for c in range(7) if (legal_int >> (c * 9)) & 0x3F]
print(f"\nAfter filling column 3:  legal columns = {open_cols}")
print(f"Column 3 open? {3 in open_cols}  (full — no bit set in C3!)")

### 3.2 N-Tuple Network

An N-Tuple Network is a sum of **Look-Up Tables (LUTs)**. Each tuple is a fixed pattern of $N$ board cells. For a given board state:

1. Read the $N$ cells — each is one of 4 states: `empty`, `yellow`, `red`, or `reachable` (empty + legal move landing)
2. Encode the $N$ cell values as a **base-4 index** $T \in [0, 4^N)$
3. Look up weight $W[T]$ in that tuple's LUT
4. **Sum** over all $M$ tuples (plus their horizontal mirrors)
5. Apply $\tanh$ to squash the output to $[-1, +1]$

$$V(s) = \tanh\!\left(\sum_{m=1}^{M} \big[W_m[T_m(s)] + W_m[T_m(\text{mirror}(s))]\big]\right)$$

| Component | Size |
|-----------|------|
| Tuples ($M$) | 200 patterns |
| Cells per tuple ($N$) | 8 |
| LUT entries per tuple ($4^N$) | 65,536 |
| Players | 2 (separate LUTs for Yellow and Red) |
| **Total parameters** | $2 \times 200 \times 65{,}536 = 26{,}214{,}400$ |

The key advantage: **no matrix multiplications** — just integer indexing into arrays. This makes both forward passes and gradient updates extremely fast.

In [ ]:
from techdays26.ntuple_network import NTupleNetwork
from techdays26.ntuples import NTUPLE_BITIDX_LIST_200, format_ntuple, ntuple_summary

info = ntuple_summary(NTUPLE_BITIDX_LIST_200)
print(f"N-Tuple set:  M={info['count']} tuples, N={info['length']} cells each")
print(f"LUT size per tuple: 4^{info['length']} = {info.get('lut_size', '?')}")

net = NTupleNetwork(NTUPLE_BITIDX_LIST_200)
n_params = sum(p.numel() for p in net.parameters())
print(f"Total parameters: {n_params:,}")

print(f"\nExample tuple #0 (cells highlighted with X):")
print(format_ntuple(NTUPLE_BITIDX_LIST_200[0]))

### 3.3 N-Tuple Visualizer

The `NTupleVisualizer` lets you load pretrained weights and inspect the value function interactively:

- **Board panel** — click columns to build positions
- **Pattern overlay** — see which cells each tuple reads (with mirror toggle)
- **LUT heatmap** — the raw weight table for the selected tuple
- **Per-move value bar** — the network's score for each legal move

Load the pretrained model below and explore how the agent evaluates different positions.

In [ ]:
%matplotlib ipympl
from techdays26.gui_ntuple import NTupleVisualizer

vis = NTupleVisualizer(
    model_path="../exp_20260228_13-46/repeat_0/step_500_model_weights.pt"
)
vis.show()

### 3.4 ε-Greedy Afterstate Policy

The agent picks moves using **afterstate evaluation**:

1. For each legal (non-losing) move, **play it on a copy** of the board
2. If the move wins → value = `+1` (terminal reward)
3. Otherwise → evaluate the **afterstate** with the value network
4. **Yellow maximises**, **Red minimises** (zero-sum)
5. With probability $\varepsilon$, pick a **random** legal move instead (exploration)

The `best_afterstate_values()` function does this for an entire batch at once, using reservoir sampling for the random moves — no extra pass needed.

```
   s ─── action a ───▶ s' (afterstate)
                        │
                  V(s') = network score
                        │
         pick a = argmax V(s')  [Yellow]
              or argmin V(s')   [Red]
```

Try the interactive version — watch the agent decide between exploitation and exploration. Drag ε to see how randomness affects move selection:

In [ ]:
%matplotlib ipympl
from techdays26.gui_epsilon import EpsilonGreedyVisualizer

vis = EpsilonGreedyVisualizer(
    model_path="../exp_20260228_13-46/repeat_0/step_500_model_weights.pt"
)
vis.show()

> **Questions — §3 Agent & Policy**
>
> 1. With 200 tuples of length 6 and 4 cell states, the network has ~1.6M parameters. A fully-connected MLP with comparable capacity would need how many weights? What is the key trade-off?
> 2. Why does the network use `tanh` as its output activation? What would happen with an unbounded output?
> 3. Yellow maximises V and Red minimises V. Why is this correct for a zero-sum game? What would need to change if rewards were not symmetric?
> 4. What happens to the agent's play quality if ε is set to 0 during training? What about during evaluation?
> 5. *(Bonus — from the legal-move discussion above)* On an empty board, how many legal moves are there? How many *distinct* afterstates do they produce?

<details>
<summary><b>💡 Answers — click to reveal</b></summary>

1. **N-Tuple vs. MLP.** A dense MLP with a single hidden layer of width $h$, fed the 42-cell board as, say, 84 one-hot inputs, has roughly $84 h + h$ parameters — so $h \approx 20{,}000$ hidden units to match 1.6M weights. The N-Tuple network has the same parameter budget but **no matrix multiplies at inference time**: you read 200 LUT entries (integer indexing) and sum them. Trade-off: N-tuples give you enormous, spatially-local capacity cheaply, but they do *not* learn distributed features the way an MLP does — the inductive bias is that local patterns matter.

2. **Why tanh.** Rewards are in $\{-1, 0, +1\}$, so the true value function is bounded in $[-1, +1]$. `tanh` enforces this bound — more importantly, it keeps the **bootstrap target** inside the same range, preventing positive-feedback loops where the online network chases its own unbounded predictions.

3. **Maximise / minimise.** Rewards are Yellow-positive: $+1$ means Yellow wins, $-1$ means Red wins. Since it is zero-sum, Red prefers whatever Yellow least prefers — i.e., Red wants to minimise Yellow's value. Both players share the *same* value function; there is no "Red's V" and "Yellow's V" to reconcile. If rewards were asymmetric (e.g., different payoffs for different win types), you would need two value functions or a sign-flipping convention that depends on the winning player.

4. **ε = 0 during training** → the agent only ever picks the greedy move, gets stuck replaying the same lines, and cannot discover new positions or correct systematic mis-evaluations. **ε = 0 during evaluation** → correct: exploration is a training trick, at eval time you want the strongest move every step.

5. **Empty-board moves.** 7 legal columns, but by left-right symmetry only **4 distinct afterstates** (cols $0{\leftrightarrow}6$, $1{\leftrightarrow}5$, $2{\leftrightarrow}4$, $3$ self-symmetric). This is one of the simplest examples of the many-to-one compression V(afterstate) gives you.

</details>

## 📉 4. Update Rule — TD(λ)

The agent improves by playing games against itself and updating its value function after every move. The update rule is **Temporal-Difference learning with eligibility traces** — TD(λ).

### 4.1 TD Error

**One training step = one half-move.** After a single move (by whichever player is to move), the agent compares **what it predicted** with **what it sees now**:

```
  before the move        after the move
  ┌──────────┐           ┌──────────┐
  │    s      │   move    │    s'     │
  │(afterstate)│──────────▶│(afterstate)│
  └─────┬─────┘           └─────┬─────┘
        │                       │
      V(s)                V_target(s')
   "what we                "what we
    predicted"              see now"

        TD error:  δ = r + V_target(s') − V(s)
```

More precisely:

$$\delta = \underbrace{r + V_{\text{target}}(s')}_{\text{TD target}} - \underbrace{V(s)}_{\text{previous estimate}}$$

- $V(s)$ — value of the afterstate *before* this move was played (stored at the start of the training step)
- $r$ — reward (`+1` / `−1` at game end, `0` otherwise)
- $V_{\text{target}}(s')$ — bootstrap from a **frozen target network** (Polyak-averaged copy of the online network), evaluated on the afterstate *after* this move

There is no separate "opponent move" hiding between $s$ and $s'$. Self-play just means the next training step happens to be the other colour's turn — but that is simply the next update, not part of this one.

For terminal states, no bootstrapping is needed: the target is simply the reward $r$.

If $\delta > 0$, the new position looks **better** than we thought — we should increase $V(s)$.  
If $\delta < 0$, it looks **worse** — we should decrease it.

---

#### 🎬 Credit assignment — watch the reward flow backward

Here is the core difficulty of reinforcement learning on sparse-reward games: the agent plays 20-40 moves, then a single $\pm 1$ signal arrives at the end. How does that one bit of information teach the value function about the *opening* move?

**The widget below plays a full game, sets $V(s_t) = 0$ at every step (the agent knows nothing), then lets you propagate the terminal reward backward one TD(0) update at a time.** Watch how many clicks it takes for the opening-move value to catch a whiff of the final outcome — that's exactly the slowdown TD(λ) with $\lambda > 0$ fixes in §4.3–4.4.


In [ ]:
%matplotlib ipympl
from techdays26.gui_credit_assignment import CreditAssignmentVisualizer

vis = CreditAssignmentVisualizer(
    model_path="../exp_20260228_13-46/repeat_0/step_500_model_weights.pt"
)
vis.show()

### 4.2 Putting It All Together — One Complete TD(0) Update

Before we make the target any smarter, let's watch **one complete update cycle** play out end-to-end in its simplest form: the **1-step TD target** (also known as **TD(0)**).

The widget below walks through every phase of a single training step — evaluate the legal afterstates, pick a move ε-greedily (exploit or explore), play it, compute the TD(0) target $r + V(s')$, and see how the weight update shrinks the error. This ties §4 (action selection) and §3.2 (TD error) into one coherent loop.

> 💡 **TD(0) is a stepping stone.** In §4.3–5.4 we will replace $V(s')$ with a smarter target — an **n-step return** and then the **truncated λ-return** used by the real training loop. Everything else in this cycle (action selection, playing the move, the gradient step) stays exactly the same; only the target changes.


In [ ]:
%matplotlib ipympl
from techdays26.gui_td_training_step import TDTrainingStepVisualizer

vis = TDTrainingStepVisualizer(
    model_path="../exp_20260228_13-46/repeat_0/step_500_model_weights.pt"
)
vis.show()

### 4.3 N-Step Returns — The Bias-Variance Spectrum

Before combining multiple horizons (TD(λ)), let's understand the building blocks: **n-step returns**.

The general **n-step return** from step $t$ sums the next $n$ rewards and then bootstraps from the value $n$ steps ahead:

$$G_t^{(n)} = r_{t+1} + r_{t+2} + \dots + r_{t+n} + V(s_{t+n}) \qquad \text{(no discount in our episodic setting)}$$

Since Connect-4 has only **sparse terminal rewards** (zero except at game end), the sum collapses:

$$G_t^{(n)} = V(s_{t+n}) \quad \text{if } t{+}n < T, \qquad G_t^{(n)} = r_T \quad \text{if } t{+}n \ge T$$

| Horizon | Return | Character |
|---------|--------|-----------|
| $n = 1$ | $G_t^{(1)} = V(s_{t+1})$ | **1-step TD** — heavy bootstrap, biased toward current $V$ estimate, low variance |
| $n = T{-}t$ | $G_t^{(T-t)} = r_T$ | **Monte Carlo** — uses actual outcome, unbiased, high variance |
| $1 < n < T{-}t$ | $G_t^{(n)} = V(s_{t+n})$ | Intermediate — trades bias vs. variance |

**Bias vs. variance:** Small $n$ relies heavily on the (possibly wrong) current value estimate → high **bias**, low variance. Large $n$ uses more real game signal → low bias, but fewer independent samples per update → high **variance**.

The key insight: **TD(λ) in the next section is just a weighted average of all these n-step returns.**

Drag the **n** slider to change the bootstrap horizon. Watch how the target shifts from the next value estimate (n=1) all the way to the actual game outcome (n=T−t):

In [ ]:
%matplotlib ipympl
from techdays26.gui_nstep import NStepReturnVisualizer

vis = NStepReturnVisualizer(
    model_path="../exp_20260228_13-46/repeat_0/step_500_model_weights.pt"
)
vis.show()

### 4.4 TD(λ) — Eligibility Traces via Truncated Returns

Plain TD (λ=0) only propagates information one step at a time — slow for a 42-move game. TD(λ) blends multi-step returns using a decay factor $\lambda \in [0, 1]$:

$$G_t^{\lambda} = (1 - \lambda) \sum_{n=1}^{\infty} \lambda^{n-1} G_t^{(n)}$$

In practice we use a **truncated forward view**: a ring buffer stores the next $k$ afterstate values looking forward from step $t$. When the buffer is full (or a game ends), we fold the λ-weighted returns back onto $V(s_t)$:

```
Step:     t    t+1    t+2    t+3   ...   t+k
Value:   V₀     V₁     V₂     V₃   ...   Vₖ
                                          ↑ bootstrap
Target for V₀:  (1-λ)·V₁ + (1-λ)·λ·V₂ + (1-λ)·λ²·V₃ + ... + λᵏ⁻¹·Vₖ
```

| Parameter | Role |
|-----------|------|
| $\lambda = 0$ | Pure one-step TD — all weight on $V(s_{t+1})$; fast but slow credit assignment |
| $\lambda = 1$ | All weight on the tail $V(s_{t+k})$. Becomes pure Monte Carlo **only** when the buffer reaches the terminal state ($t{+}k = T$); otherwise it still bootstraps from $V(s_{t+k})$ |
| $\lambda \approx 0.5$ | Good balance for Connect-4 (our default) |
| $k$ (`n_truncate`) | Ring-buffer size — how many **future** steps we look ahead before folding the λ-weighted sum back onto $V(s_t)$ |

Drag the sliders to build intuition for how λ and the truncation horizon affect the weight distribution:

In [ ]:
%matplotlib ipympl
from techdays26.gui_lambda import LambdaReturnVisualizer

vis = LambdaReturnVisualizer()
vis.show()

The continuous view below shows the same weights as a **decay envelope**. Notice: when λ is high, the terminal bar (actual return at T) receives the remaining mass λ^(T−t−1), which can be **larger** than the preceding bars — the integral of the geometric tail concentrates at the end:

In [ ]:
%matplotlib ipympl
from techdays26.gui_lambda_decay import LambdaDecayVisualizer

vis = LambdaDecayVisualizer()
vis.show()

### 4.5 Gradient Step

Given the λ-return target $G^\lambda$ for each position in the buffer, the loss is **mean squared error**:

$$\mathcal{L} = \frac{1}{|\mathcal{B}|} \sum_{b \in \mathcal{B}} \big(V_\theta(s_b) - G_b^\lambda\big)^2$$

Moves that were chosen randomly (ε-exploration) are **excluded** from the loss — they provide diverse experience but shouldn't drive the gradient, since their afterstates weren't selected by the current policy.

**Optimiser**: Adam with exponential LR decay from `lr_initial` to `lr_final`.

**Target network**: a Polyak-averaged copy updated each step via:

$$\theta_{\text{target}} \leftarrow \tau \cdot \theta + (1 - \tau) \cdot \theta_{\text{target}}$$

This stabilises the bootstrap targets and prevents oscillation.

> The full training loop is implemented in **[`torch_ntuple_net_lambda.ipynb`](../torch_ntuple_net_lambda.ipynb)** §4.

Back in §4.2 you watched a single training step with a **TD(0)** target — the next afterstate's value. The widget below applies the exact same update cycle across a **full game**, but now using the **truncated λ-return** as the target at every step. Drag the step slider to move through the game; the green shading on the V-timeline shows which future values contribute to the update target at the selected step, and the λ slider lets you interpolate between TD(0) (λ=0) and Monte Carlo (λ=1).

In [ ]:
%matplotlib ipympl
from techdays26.gui_td_step import TDStepVisualizer

vis = TDStepVisualizer(
    model_path="../exp_20260228_13-46/repeat_0/step_500_model_weights.pt"
)
vis.show()

> **Questions — §4 Update**
>
> 1. Why do we use a **target network** instead of bootstrapping directly from the online network? What kind of instability could arise without it?
> 2. TD(0) propagates information one step per update. In a 42-move Connect-4 game, roughly how many complete games would the agent need to propagate the terminal reward back to the opening move? How does λ > 0 help?
> 3. Random exploration moves are excluded from the loss. Why? What would happen if we included them?
> 4. The Polyak update uses a parameter τ. What happens at the extremes τ = 0 and τ = 1?

<details>
<summary><b>💡 Answers — click to reveal</b></summary>

1. **Target network.** Bootstrapping directly from the online network creates a **moving target**: the weight update changes $V$, which changes the target, which changes the next update — a tight positive-feedback loop that easily oscillates or diverges. A Polyak-averaged target network moves slowly, so during any single update the target looks (approximately) frozen — the gradient step is then effectively a supervised regression toward a stable target.

2. **Propagation speed.** With plain TD(0), each game only *definitely* propagates reward information one step backward (the rest relies on subsequent updates chaining). For a 42-move game you therefore need on the order of **~42 full games** to get the terminal signal all the way to move 1 — and far more in practice, because each intermediate $V$ also has to converge. TD(λ) with $\lambda > 0$ propagates information over roughly $1/(1-\lambda)$ steps per update, so $\lambda = 0.5$ gives you $\sim 2$-step propagation and $\lambda = 0.9$ gives you $\sim 10$-step propagation — a massive speedup in credit assignment.

3. **Excluding random moves.** Random exploration moves are *not* drawn from the current policy, so their afterstates are not representative of what the policy would actually reach. Training $V$ on them would bias the value function toward positions the agent never plans to visit — the network would waste capacity getting positions *right* that it intends to avoid. Including them also contaminates the bootstrap: $V(s_{t+1})$ would reflect a random action, not the agent's best response.

4. **τ extremes.** $\tau = 0$ means the target network never updates — it stays frozen at initialisation forever, and the online network chases a stale target. $\tau = 1$ means the target network is replaced by the online network every step — you lose all stabilisation and reduce to plain online bootstrapping. Good values sit close to $0$ (e.g., $\tau = 10^{-3}$): noticeable lag, but eventually tracks the online network.

</details>

## 🏆 5. Does It Actually Work?

You have walked every line of the TD(λ) learning cycle: state, policy, afterstate lookup, 1-step target, λ-return, gradient step. Time to cash the check — **did any of it stick?**

In [ ]:
%matplotlib ipympl
from bitbully import BitBully
from bitbully.gui_c4 import GuiC4
from techdays26.td_agent import TDConnect4AgentTorch

# Trained N-Tuple TD(λ) agent (no search) vs. 2-ply BitBully (shallow search)
ntuple_agent = TDConnect4AgentTorch(
    model_path="../exp_20260228_13-46/repeat_0/step_500_model_weights.pt",
)
agents = {
    "N-Tuple TD(λ)": ntuple_agent,
    "BitBully (2-ply)": BitBully(opening_book=None, tie_break="center", max_depth=2),
    "BitBully (4-ply)": BitBully(opening_book=None, tie_break="center", max_depth=4),
}
GuiC4(agents=agents, autoplay=True).get_widget()

---

## ✅ Summary

You now have a complete picture of every building block behind our Connect-4 TD(λ) agent:

| Section | Key idea |
|---|---|
| **§1 Environment** | The board lives in two 64-bit integers; all operations (play, win-check, move generation) are O(1) bitwise ops that run on GPU across thousands of games in parallel |
| **§2 State** | Three integers fully describe any position; rewards are sparse (+1/−1 at game end only); afterstates compress (state, action) pairs into a single position |
| **§3 Agent & Policy** | An N-Tuple Network sums look-up table weights — no matrix multiplies; the agent scores each legal afterstate and picks the best with ε-greedy exploration |
| **§4 Update** | TD(λ) blends n-step returns via exponentially decaying weights, trading off bootstrap **bias** (high for small $n$) against sampling **variance** (high for large $n$) |
| **§4 Does it work?** | The trained checkpoint sharply beats an untrained one and holds its own against a perfect BitBully search — all learned from self-play alone |

**Next step →** Open **[`1_train_ntuple_net.ipynb`](1_train_ntuple_net.ipynb)** to train the agent from scratch and watch it go from random play to strong Connect-4!